# 复习流程演示

这个 notebook 更贴近你现在真正想看的使用方式，不是偏底层调试，而是偏“学生复习会怎么走”。

你可以在这里测试两种场景：

- 先复习知识点，再练题
- 先做题，再看知识点

同时也能模拟学生点击这些按钮之后，下一轮推荐会怎么变化：

- `掌握很熟练`
- `还要练题`
- `这题做对了`
- `这题没做对`
- `部分会做`


In [14]:
import copy
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path('/Users/xumuchi/Desktop/TeachAgent')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from review_bundle_builder import build_review_bundles, load_example_map, load_leaf_card_lookup
from review_scheduler import load_review_state, now_from_value
from review_state_manager import apply_review_action

SCRATCH_DIR = PROJECT_ROOT / 'scratch' / 'review_session_demo'
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

REVIEW_STATE_PATH = PROJECT_ROOT / 'scratch' / 'student_annotation_merged_review_state.json'
EXAMPLES_MD_PATH = PROJECT_ROOT / 'docs' / 'rag_samples' / 'taizhou_simulated_exam_examples_batch_01.md'
DEMO_NOW = now_from_value('2026-06-21T10:00:00+08:00')

SESSION_OUTPUT_PATH = SCRATCH_DIR / 'session_current.json'
AFTER_NODE_OUTPUT_PATH = SCRATCH_DIR / 'after_node_feedback.json'
AFTER_QUESTION_OUTPUT_PATH = SCRATCH_DIR / 'after_question_feedback.json'


## 1. 载入当前学生状态

如果你之后有新的学生复习数据，只需要改这里的 `REVIEW_STATE_PATH`。

In [2]:
REVIEW_STATE = load_review_state(REVIEW_STATE_PATH)
EXAMPLE_MAP = load_example_map(EXAMPLES_MD_PATH)
LEAF_CARD_LOOKUP = load_leaf_card_lookup()

print('当前 record_id:', REVIEW_STATE.get('record_id'))
print('知识点数量:', len(REVIEW_STATE.get('knowledge_point_states', [])))
print('题目数量:', len(REVIEW_STATE.get('example_question_states', [])))

当前 record_id: review_seed.taizhou_simulated_exam_examples_batch_01.v1
知识点数量: 16
题目数量: 9


## 2. 选择复习模式

把这里改成：

- `'leaf_first'`：先知识点后题目
- `'question_first'`：先题目后知识点


In [3]:
REVIEW_MODE = 'leaf_first'  # 可改成 'question_first'

SESSION_RESULT = build_review_bundles(
    REVIEW_STATE,
    example_map=EXAMPLE_MAP,
    leaf_card_lookup=LEAF_CARD_LOOKUP,
    now=DEMO_NOW,
    mode=REVIEW_MODE,
    bundle_limit=5,
    bundle_question_limit=3,
)

SESSION_OUTPUT_PATH.write_text(
    json.dumps(SESSION_RESULT.as_dict(), ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

print('当前模式：', REVIEW_MODE)
print('复习包输出：', SESSION_OUTPUT_PATH)
print('一共生成 bundle 数量：', len(SESSION_RESULT.review_bundles))

当前模式： leaf_first
复习包输出： /Users/xumuchi/Desktop/TeachAgent/scratch/review_session_demo/session_current.json
一共生成 bundle 数量： 5


## 3. 看第一页推荐

这里直接看第一个复习包，相当于学生打开系统后最先看到的内容。

In [5]:
FIRST_BUNDLE = SESSION_RESULT.review_bundles[0]
print(json.dumps(FIRST_BUNDLE, ensure_ascii=False, indent=2))

{
  "bundle_id": "bundle_01",
  "mode": "leaf_first",
  "rank": 1,
  "node_id": "math.三角函数_平面向量与解三角形.三角函数.公式体系.三角函数诱导公式",
  "priority_score": 0.8975,
  "bundle_reason": "知识点已到复习时间，并联动 2 道练习题一起复习",
  "leaf_card": {
    "chunk_id": "chunk.math.三角函数_平面向量与解三角形.三角函数.公式体系.三角函数诱导公式.formula_card.v1",
    "node_id": "math.三角函数_平面向量与解三角形.三角函数.公式体系.三角函数诱导公式",
    "title": "三角函数诱导公式",
    "card_type": "formula_card",
    "node_kind": "formula",
    "path": [
      "数学",
      "三角函数、平面向量与解三角形",
      "三角函数",
      "公式体系",
      "三角函数诱导公式"
    ],
    "keywords": [
      "诱导公式",
      "奇偶性",
      "周期性",
      "终边关系"
    ],
    "text": "知识点：三角函数诱导公式。路径：数学 > 三角函数、平面向量与解三角形 > 三角函数 > 公式体系 > 三角函数诱导公式。别名：诱导变换公式。关键词：诱导公式；奇偶性；周期性；终边关系。公式：常见诱导公式包括 sin(-α)=-sinα，cos(-α)=cosα，sin(π-α)=sinα，cos(π-α)=-cosα，sin(π+α)=-sinα，cos(2kπ+α)=cosα 等。适用条件：角可写成 ±α、π±α、2kπ±α、π/2±α 等标准变形；需要把复杂角转成熟悉锐角或基本角。特殊情况：口诀常概括为“奇变偶不变，符号看象限”，但最终仍应回到单位圆或终边关系核对；不同变形对应保留原函数名或变换函数名要分清。变量说明：α 通常取参考角，k 为整数；公式使用时要同时判断函数名变化和符号变化。推导提示：诱导公式本质上来自终边关于

## 4. 如果是知识点模式，先看卡片，再模拟学生点击知识点反馈

这里测试两个常见按钮：

- `掌握很熟练`
- `还要练题`


In [8]:
if FIRST_BUNDLE['mode'] == 'leaf_first':
    TARGET_NODE_ID = FIRST_BUNDLE['node_id']
    print('当前知识点：', TARGET_NODE_ID)
    print('知识点标题：', FIRST_BUNDLE['leaf_card']['title'] if FIRST_BUNDLE.get('leaf_card') else '无')
    print('当前配套题数：', FIRST_BUNDLE['question_count'])
else:
    print('当前不是知识点模式，请把 REVIEW_MODE 改成 leaf_first 再运行这一格。')

当前知识点： math.三角函数_平面向量与解三角形.三角函数.公式体系.三角函数诱导公式
知识点标题： 三角函数诱导公式
当前配套题数： 2


In [9]:
if FIRST_BUNDLE['mode'] == 'leaf_first':
    node_event = apply_review_action(
        copy.deepcopy(REVIEW_STATE),
        action='node_needs_more_practice',  # 可改成 'node_mastered_well'
        target_type='knowledge_point',
        target_id=FIRST_BUNDLE['node_id'],
        now=DEMO_NOW,
    )
    AFTER_NODE_OUTPUT_PATH.write_text(
        json.dumps(node_event.as_dict(), ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    print('知识点反馈结果已写入：', AFTER_NODE_OUTPUT_PATH)
    print('动作：', node_event.action)
    print('目标：', node_event.target_id)
else:
    print('当前不是知识点模式。')

知识点反馈结果已写入： /Users/xumuchi/Desktop/TeachAgent/scratch/review_session_demo/after_node_feedback.json
动作： node_needs_more_practice
目标： math.三角函数_平面向量与解三角形.三角函数.公式体系.三角函数诱导公式


## 5. 知识点反馈后，再生成下一轮推荐

这样就能看到：

- 如果点了 `还要练题`，相关题目会不会更靠前
- 如果点了 `掌握很熟练`，该知识点会不会后移


In [10]:
if FIRST_BUNDLE['mode'] == 'leaf_first':
    NEXT_AFTER_NODE = build_review_bundles(
        node_event.updated_payload,
        example_map=EXAMPLE_MAP,
        leaf_card_lookup=LEAF_CARD_LOOKUP,
        now=DEMO_NOW,
        mode='leaf_first',
        bundle_limit=5,
        bundle_question_limit=3,
    )
    print('知识点反馈后的 Top-3 bundle：')
    for bundle in NEXT_AFTER_NODE.review_bundles[:3]:
        title = bundle['leaf_card']['title'] if bundle.get('leaf_card') else bundle.get('node_id')
        print('-', title, '| 题数 =', bundle.get('question_count'))
else:
    print('当前不是知识点模式。')

知识点反馈后的 Top-3 bundle：
- 函数图象平移变换 | 题数 = 2
- 向量夹角公式 | 题数 = 2
- 数量积几何意义 | 题数 = 2


## 6. 如果是题目模式，模拟学生做题反馈

这里测试：

- `correct`
- `wrong`
- `partial`


In [11]:
if FIRST_BUNDLE['mode'] == 'question_first':
    TARGET_QUESTION_ID = FIRST_BUNDLE['question']['question_id']
    print('当前题目：', TARGET_QUESTION_ID)
    print('题干：', (FIRST_BUNDLE['question']['content'].get('stem') or '')[:120])
else:
    print('当前不是题目模式，请把 REVIEW_MODE 改成 question_first 再运行这一格。')

当前不是题目模式，请把 REVIEW_MODE 改成 question_first 再运行这一格。


In [12]:
if FIRST_BUNDLE['mode'] == 'question_first':
    question_event = apply_review_action(
        copy.deepcopy(REVIEW_STATE),
        action='review_result',
        target_type='wrong_question',
        target_id=FIRST_BUNDLE['question']['question_id'],
        result='wrong',  # 可改成 'correct' 或 'partial'
        now=DEMO_NOW,
    )
    AFTER_QUESTION_OUTPUT_PATH.write_text(
        json.dumps(question_event.as_dict(), ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    print('题目反馈结果已写入：', AFTER_QUESTION_OUTPUT_PATH)
    print('动作：', question_event.action)
    print('题目：', question_event.target_id)
else:
    print('当前不是题目模式。')

当前不是题目模式。


## 7. 题目反馈后，再看下一轮推荐

这个最适合观察：做错一道题后，它绑定的知识点会不会一起升上来。

In [13]:
if FIRST_BUNDLE['mode'] == 'question_first':
    NEXT_AFTER_QUESTION = build_review_bundles(
        question_event.updated_payload,
        example_map=EXAMPLE_MAP,
        leaf_card_lookup=LEAF_CARD_LOOKUP,
        now=DEMO_NOW,
        mode='question_first',
        bundle_limit=5,
        bundle_question_limit=3,
    )
    print('题目反馈后的 Top-3 bundle：')
    for bundle in NEXT_AFTER_QUESTION.review_bundles[:3]:
        print('-', bundle['question']['question_id'], '| 关联知识点数 =', len(bundle.get('linked_leaf_cards', [])))
else:
    print('当前不是题目模式。')

当前不是题目模式。
